In [ ]:
import xarray as xr
import pandas as pd
import xskillscore as xs
import matplotlib.pyplot as plt

import albedo_functions as af

from pathlib import Path

import logging

from concurrent.futures import ProcessPoolExecutor

In [ ]:
# Configurazione logging
log_file = "process_log.txt"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[
        logging.FileHandler(log_file),  # Write to file
    ]
)

# Percorso ai dati
DATA_PATH = Path("/confess/dicarlo/02-confess/05-confess-post-data")
OUTPUT_PATH = Path("/confess/dicarlo/02-confess/figures_2025")
OBS_PATH = Path("/confess/dicarlo/cartella_ordinata")

# Esperimenti e variabili
exp_sens = "a52o"
exp_ctrl = "a1ua"
var = "tas"

lead_number=3

In [ ]:
# Caricamento dataset di controllo
dset_ctrl_path = DATA_PATH / exp_ctrl / "1x1" / var / f"{exp_ctrl}_{var}_Amon_EC-Earth3_dcppA-hindcast_lead_3-4_1x1_ensemble_rad.nc"
dset_ctrl = xr.open_dataset(dset_ctrl_path)
dset_ctrl['time'] = pd.to_datetime(dset_ctrl['time'].values).normalize().to_period('M').start_time

# Caricamento dataset sensibile
dset_sens_path = DATA_PATH / exp_sens / "1x1" / var / f"{exp_sens}_{var}_Amon_EC-Earth3_dcppA-hindcast_lead_3-4_1x1_ensemble_rad.nc"
dset_sens = xr.open_dataset(dset_sens_path)
dset_sens['time'] = pd.to_datetime(dset_sens['time'].values).normalize().to_period('M').start_time
dset_sens = dset_sens.sel(time=slice(dset_ctrl.time[6], dset_ctrl.time[-1]))

# Caricamento osservazioni
obs_path = OBS_PATH / f"ERA5_2t_1x1_2year.nc"
obs = xr.open_dataset(obs_path)
obs['time'] = pd.to_datetime(obs['time'].values) - pd.DateOffset(months=1)
obs['time'] = pd.to_datetime(obs['time'].values).normalize().to_period('M').start_time
obs = obs.rename({'2t': 'tas'})
obs = obs.sel(time=slice(dset_ctrl.time[6], dset_ctrl.time[-1]))

dset_ctrl = dset_ctrl.sel(time=slice(dset_ctrl.time[6], dset_ctrl.time[-1]))
#    # Calcolo anomalies
dset_ctrl_anom = dset_ctrl - dset_ctrl.mean('time')
dset_sens_anom = dset_sens - dset_sens.mean('time')
obs_anom = obs - obs.mean('time')

# Calcolo correlazioni
dset_ctrl_acc, dset_ctrl_p = af.alb_corr(obs_anom[f"{var}"], dset_ctrl_anom[f"{var}"])
dset_sens_acc, dset_sens_p = af.alb_corr(obs_anom[f"{var}"], dset_sens_anom[f"{var}"])

# Conversione in Dataset
dset_ctrl_acc_ = dset_ctrl_acc.to_dataset(name=f"{var}")
dset_ctrl_p_ = dset_ctrl_p.to_dataset(name=f"{var}")
    
dset_sens_acc_ = dset_sens_acc.to_dataset(name=f"{var}")
dset_sens_p_ = dset_sens_p.to_dataset(name=f"{var}")

sig_delta = af.bootstrap_quantile_correlation(dset_sens_anom, dset_ctrl_anom, obs_anom)
#sig_delta_ = sig_delta.to_dataset(name=f"{var}")

In [ ]:
dset_ctrl_acc_.to_netcdf(f"{OBS_PATH}/{exp_ctrl}_tas_ACC_r_3-4_1999.nc")
dset_ctrl_p_.to_netcdf(f"{OBS_PATH}/{exp_ctrl}_tas_ACC_p_3-4_1999.nc")
dset_sens_acc_.to_netcdf(f"{OBS_PATH}/{exp_sens}_tas_ACC_r_3-4_1999.nc")
dset_sens_p_.to_netcdf(f"{OBS_PATH}/{exp_sens}_tas_ACC_p_3-4_1999.nc")
sig_delta.to_netcdf(f"{OBS_PATH}/{exp_sens}-{exp_ctrl}_tas_bootstrap_3-4_1999.nc")